# Phase 0 — Reference stack + nondeterminism reproduction

DetKernels Phase 0: confirm GPU, run a reference inference stack (vLLM) on a small
open model, and reproduce the core claim from "Defeating Nondeterminism in LLM
Inference" — that outputs diverge across batch sizes even under greedy decoding.

Run this top-to-bottom on a Colab GPU runtime (Runtime > Change runtime type > GPU).

At the end this saves `report_phase0.json` with:
- a baseline sanity check (repeated runs at fixed batch size 1 should already be
  bitwise identical)
- repeated runs at batch size 32 (should NOT be identical if the claim holds)
- the first token index where a batch-size-1 run diverges from a batch-size-32 run
  of the same prompt

Download that file and share it back so it can feed the Phase 0 gate decision and
the Phase 1 harness design.

In [ ]:
!nvidia-smi

In [ ]:
# vLLM's default PyPI wheel (v0.20+) targets the CUDA 13.0 runtime, which
# doesn't match Colab's CUDA 12.x toolkit and surfaces as "ImportError:
# libcudart.so.13: cannot open shared object file" on `import vllm`. Attempts to
# fix this via `--torch-backend=cu129` were unreliable (see
# https://github.com/vllm-project/vllm/issues/43435, closed as not planned).
# Confirmed working fix: pin to just before the CUDA-13-default switch.
!pip install -q "vllm<0.20" transformers accelerate openai requests

**If Colab prompts you to restart the runtime after the install above, do so, then
continue from the cell below (no need to re-run the install cell).**

In [ ]:
import json
import subprocess
import time

import requests

MODEL = "Qwen/Qwen3-1.7B"  # small model to keep iteration fast, per project.md Phase 0
PROMPT = (
    "Explain in one paragraph why floating point addition is not associative, "
    "and why that matters for reproducibility."
)
MAX_TOKENS = 64
N_REPEATS = 20  # bump toward 100+ once this runs cleanly, per Phase 1 harness plan
PORT = 8000

# Run vLLM as a genuine background server process instead of embedding the LLM
# class directly in this notebook's Python process. The embedded approach hit
# "io.UnsupportedOperation: fileno" repeatedly in Jupyter/Colab, because several
# independent parts of vLLM's startup path (the V1 multiprocessing engine core,
# torch.compile's output capture, and post-NCCL-init logging) each separately
# assume a real OS file descriptor for stdout, which ipykernel's fake stream
# doesn't provide. A subprocess gets a genuine stdout (redirected to a log file
# below), sidestepping all of those failure points at once.
#
# float16, not bfloat16: Colab's free-tier T4 GPUs are compute capability 7.5,
# which lacks bf16 support (Ampere/8.0+ only).
log_file = open("vllm_server.log", "w")
server_proc = subprocess.Popen(
    [
        "vllm", "serve", MODEL,
        "--dtype", "float16",
        "--gpu-memory-utilization", "0.85",
        "--port", str(PORT),
        "--seed", "0",
    ],
    stdout=log_file,
    stderr=subprocess.STDOUT,
)

print(f"Started vLLM server (pid={server_proc.pid}), waiting for it to become healthy...")
for _ in range(120):
    if server_proc.poll() is not None:
        raise RuntimeError(
            f"vLLM server exited early with code {server_proc.returncode}; "
            "check vllm_server.log for the traceback."
        )
    try:
        r = requests.get(f"http://localhost:{PORT}/health", timeout=2)
        if r.status_code == 200:
            print("Server is healthy.")
            break
    except requests.exceptions.ConnectionError:
        pass
    time.sleep(5)
else:
    raise RuntimeError("Server did not become healthy in time; check vllm_server.log")

In [ ]:
import asyncio

from openai import AsyncOpenAI

client = AsyncOpenAI(base_url=f"http://localhost:{PORT}/v1", api_key="EMPTY")


async def single_request(prompt: str):
    resp = await client.completions.create(
        model=MODEL,
        prompt=prompt,
        max_tokens=MAX_TOKENS,
        temperature=0.0,
        logprobs=0,  # ask the server to return per-token strings we can diff
    )
    choice = resp.choices[0]
    return tuple(choice.logprobs.tokens) if choice.logprobs else (choice.text,)


async def run_batch(prompt: str, batch_size: int):
    """Fire `batch_size` requests concurrently so vLLM's scheduler batches them
    together; return the token sequence of the FIRST completed request. Firing
    them concurrently (not sequentially) is what changes the server's internal
    reduction order, not the prompt content."""
    tasks = [single_request(prompt) for _ in range(batch_size)]
    results = await asyncio.gather(*tasks)
    return results[0]


def first_divergence(seq_a, seq_b):
    for i, (a, b) in enumerate(zip(seq_a, seq_b)):
        if a != b:
            return i
    if len(seq_a) != len(seq_b):
        return min(len(seq_a), len(seq_b))
    return None  # identical

In [ ]:
# Baseline sanity check: fixed batch size 1, repeated SEQUENTIALLY (not
# concurrently) so each repeat really is served alone with nothing else in
# flight — gathering repeats concurrently would let them batch with each other
# and defeat the "batch size 1" baseline. Should already be deterministic — if
# this ISN'T identical across runs, something other than batch-size-dependent
# reduction order is at play (e.g. nondeterministic CUDA kernels regardless of
# batch, or a sampling bug) and that needs to be run down before trusting any
# batch-size comparison below.
bs1_runs = []
for _ in range(N_REPEATS):
    bs1_runs.append(await run_batch(PROMPT, batch_size=1))
bs1_all_identical = all(r == bs1_runs[0] for r in bs1_runs)
print("batch_size=1 repeated runs all identical:", bs1_all_identical)

In [ ]:
# Repeated runs at batch size 32 — this is the condition expected to reveal
# nondeterminism per the Thinking Machines / SGLang writeups. Each repeat's 32
# requests are fired concurrently via asyncio.gather (so they land in the same
# server-side batch); repeats themselves run sequentially so they don't bleed
# into each other.
bs32_runs = []
for _ in range(N_REPEATS):
    bs32_runs.append(await run_batch(PROMPT, batch_size=32))
bs32_all_identical = all(r == bs32_runs[0] for r in bs32_runs)
print("batch_size=32 repeated runs all identical:", bs32_all_identical)

bs32_divergence_points = []
for i in range(1, len(bs32_runs)):
    d = first_divergence(bs32_runs[0], bs32_runs[i])
    if d is not None:
        bs32_divergence_points.append(d)
print("first-divergence token indices vs run 0:", bs32_divergence_points)

In [ ]:
# Cross comparison: batch_size=1 output vs a batch_size=32 output for the SAME
# prompt. This is the direct batch-size-dependent divergence the project targets.
cross_divergence = first_divergence(bs1_runs[0], bs32_runs[0])
print("first token index where bs=1 output diverges from bs=32 output:", cross_divergence)

In [ ]:
report = {
    "model": MODEL,
    "prompt": PROMPT,
    "max_tokens": MAX_TOKENS,
    "n_repeats": N_REPEATS,
    "timestamp": time.strftime("%Y-%m-%dT%H:%M:%S"),
    "baseline_bs1_all_identical": bs1_all_identical,
    "bs32_all_identical": bs32_all_identical,
    "bs32_first_divergence_token_indices": bs32_divergence_points,
    "bs1_vs_bs32_first_divergence_token_index": cross_divergence,
    "bs1_run0_tokens": list(bs1_runs[0]),
    "bs32_run0_tokens": list(bs32_runs[0]),
}

with open("report_phase0.json", "w") as f:
    json.dump(report, f, indent=2)

print(json.dumps(report, indent=2))

In [ ]:
# Cleanup: stop the background server and free the GPU memory it's holding.
# Safe to skip if you want to keep experimenting against the same server.
server_proc.terminate()
server_proc.wait(timeout=30)
log_file.close()
print("Server stopped.")

## Next steps

Download `report_phase0.json` (Files pane on the left, or `from google.colab import
files; files.download("report_phase0.json")`) and bring it back to the main repo.

If `bs32_all_identical` is `True` and `bs1_vs_bs32_first_divergence_token_index` is
`None`, the stock stack didn't show the expected nondeterminism with these
settings — try a longer `MAX_TOKENS`, a different prompt, or check the installed
vLLM version against what the original writeups tested against, before concluding
the claim doesn't reproduce.